In [35]:
import math

pi = math.pi

def w(P, I): # Number of points of P inside any interval I over the circle.
    w = 0
    for p in P:
        if I[0] <= p <= I[1]:
            w = w + 1
        elif (I[0] >= I[1]) and (p >= I[0] or p <= I[1]):
            w = w + 1
    return w


def assign(P,A): # Given an array of intervals, A over the circle, it assigns index and the weight function defined before to every element of A
    S = []
    i = 0
    for a in A:
        S.append([a,i,w(P,a)])
        i = i+1
    return S


def SelectIntervals(A, p): # For a fixed point p this function collects intervals from a collection of intervals A lying over p on line
    I_p = []
    for I in A:
        if I[0][0] <= p <= I[0][1]:
            I_p.append(I)
    return I_p


def FirstPoint(P, I): # Given P and a interval, I over a line. This finds the first point of I from the set P 
    i = 0
    for p in P:
        i = i + 1
        if I[0] <= p <= I[1]:
            return i


def MinCoverSum(P, A): # Given the set of points P and set of intervals A (with weights and indices) over a line, this finds the minimum weight cover of P using DP
    C = []
    for p_i in P:
        C.append([p_i, SelectIntervals(A, p_i)])
    DP = []
    Sol = []
    DP.append(0)
    Sol.append([])
    for i in range(1, len(P) + 1):
        DP.append(float('Inf'))
        Sol.append([])
        for A_j in C[i - 1][1]:
            g = FirstPoint(P, A_j[0])
            if DP[g-1] + A_j[2] < DP[i]:
                DP[i] = DP[g-1] + A_j[2]
                Sol[i] = Sol[g-1][:]
                Sol[i].append(A_j)
    return Sol[len(P)]


def Tl(P,A,s): # Translate a system of points P and intervals I to 0 by shifting everything by a factor of first point of P 
    S = [[0] * len(P), [[[0, 0], 0, 0] for _ in I]]
    i = 0
    for p in P: # Translate Points
        if p >= s:
            S[0][i] = p-s
        else:
            S[0][i] = 2*pi+(p-s)
        i+=1
    i = 0
    for K in A: # Translate Intervals
        if K[0][0] >= s:
            S[1][i][0][0] = K[0][0]-s
        else:
            S[1][i][0][0] = 2*pi+ K[0][0]-s
        if K[0][1] >= s:
            S[1][i][0][1] = K[0][1]-s
        else:
            S[1][i][0][1] = 2*pi+ K[0][1]-s
        S[1][i][1] = K[1]
        S[1][i][2] = K[2]
        i+=1
    return S

def Big(P,I): # It takes in the initial set of points and intervals and outputs all the points and intervals we need to do the linear DP on
    n = len(P)
    m = len(I)
    A = assign(P,I)
    T = Tl(P,A,P[0])
    S = []
    for j in range(m):
        if T[1][j][0][0]>T[1][j][0][1]:
            S.append(j)
    #k = number of intervals over P[0]
    k = len(S)
    BP = [[] for _ in range(k)]
    BI = [[] for _ in range(k)]
    for j in range(k):
        for i in range(n):
            if T[0][i]>T[1][S[j]][0][1] and T[0][i]<T[1][S[j]][0][0]:
                BP[j].append(T[0][i])
        for l in range(m):
            if (not(T[1][l][0][1] >= T[1][S[j]][0][1] and T[1][l][0][0] <= T[1][S[j]][0][0] and T[1][l][0][1] < T[1][l][0][0])) and (not(T[1][S[j]][0][1] >= T[1][l][0][1] and T[1][S[j]][0][0] <= T[1][l][0][0])) and (not(T[1][S[j]][0][1] >= T[1][l][0][1] and T[1][l][0][0] <= T[1][l][0][1])) and (not(T[1][l][0][1] >= T[1][l][0][1] and T[1][S[j]][0][0] <= T[1][l][0][0])):
                if T[1][l][0][0] > T[1][S[j]][0][1] and T[1][l][0][1] < T[1][S[j]][0][0] and T[1][l][0][1] > T[1][l][0][0]:
                    BI[j].append(T[1][l])
                elif T[1][l][0][1] > T[1][S[j]][0][1] and (T[1][l][0][0] <= T[1][S[j]][0][1] or T[1][S[j]][0][0] <= T[1][l][0][0]):
                    BI[j].append([[T[1][S[j]][0][1],T[1][l][0][1]],T[1][l][1],T[1][l][2]])
                elif T[1][S[j]][0][0] >= T[1][l][0][0] and (T[1][l][0][1] >= T[1][S[j]][0][0] or T[1][l][0][1] <= T[1][S[j]][0][1]):
                    BI[j].append([[T[1][l][0][0],T[1][S[j]][0][0]],T[1][l][1],T[1][l][2]])
    return [BP, BI]

def Soln(P,I): # It outputs the possible solutions that come from the linear DP
    k = 0
    m = len(I)
    A = assign(P,I)
    T = Tl(P,A,P[0])
    S = []
    for j in range(m):
        if T[1][j][0][0]>T[1][j][0][1]:
            S.append(j)
    k = len(S)
    C = Big(P,I)
    L = []
    for i in range(k):
        M = [S[i]]
        M[len(M):] = ([x[1] for x in MinCoverSum(C[0][i],C[1][i])])
        L.append(M)
    return L    

def Check(P,A): # checks if the input is correct or not 
    for a in A:
        if a[0]==a[1]:
            print(a)
            return False
    j = len(P)
    for p in P:
        i = 0
        for I in A:
            if w([p],I) > 0:
                i = 1
        if i:
            j = j-1
    if j != 0:
        print("Wrong input, the intervals do not cover all the points")
        return False
    return True      


def DynamicalProgrammingOnCircle(P,I): # It gives the actual solution
    if Check(P,I):
        A = assign(P,I)
        L = Soln(P,I)
        S = []
        for a in L:
            s = 0
            for i in a:
                s += A[i][2]
            S.append(s)
        m = L[S.index(min(S))]
        K = []
        for i in m:
            K.append(I[i])
        return K

In [38]:
P = [
    0.08, 0.31, 0.57, 0.82, 1.04, 1.29,
    1.61, 1.84, 2.13, 2.39, 2.62, 2.91,
    3.17, 3.42, 3.68, 3.96, 4.21, 4.47,
    4.73, 4.98, 5.24, 5.51, 5.79, 6.08
]

I = [
    [0.20, 1.61],
    [0.82, 2.39],
    [1.29, 2.91],
    [2.13, 3.68],
    [2.62, 4.21],
    [3.17, 4.73],
    [3.96, 5.51],
    [4.47, 6.08],
    [5.24, 0.57],
    [5.79, 1.04],
    [0.08, 2.62],
    [1.84, 4.47],
    [3.42, 5.79],
    [0.57, 3.17],
    [2.91, 5.24],
    [4.98, 0.31]
]

DynamicalProgrammingOnCircle(P,I)

[[4.98, 0.31], [0.57, 3.17], [3.17, 4.73]]

In [55]:
P = [2.5]
I = [[3,2],[1,1]]
DynamicalProgrammingOnCircle(P,I)

[1, 1]


In [54]:
P = [0,2.5]
I = [[3,2]]
DynamicalProgrammingOnCircle(P,I)

Wrong input the intervals do not cover all the points
